In [3]:
from tqdm import tqdm
from json_repair import repair_json
import os
import json
import random
import re

def readListFromFile(path: str) -> list:
    """从文件中逐行读取内容到列表"""
    lines = []
    try:
        with open(path, 'r', encoding='utf-8') as f:
            lines = [line.strip() for line in f]
    except UnicodeDecodeError:
        print(f"{path} utf-8 打开失败，尝试使用gbk")
        try:
            with open(path, 'r', encoding='gbk') as f:
                lines = [line.strip() for line in f]
        except Exception as e:
            print(f"读取文件失败: {e}")
    except FileNotFoundError:
        print(f"错误: 文件未找到 - {path}")
    
    if path.endswith('.jsonl'):
        try:
            json_lines = [json.loads(line) for line in lines]
            return json_lines
        except Exception as e:
            print(f"JSON 解析失败: {e}")
            
    return lines

def saveLstAsFile(data_list: list, path: str, overwrite: bool = False):
    """将列表内容逐行写入文件"""
    if len(data_list) == 0:
        print("列表为空，无法写入文件")
        return
    
    if isinstance(data_list[0],dict):
        data_list = [json.dumps(item, ensure_ascii=False) for item in data_list]

    if os.path.exists(path) and not overwrite:
        new_path = f'{path}_{int(time.time())}.bak'
        print(f'{path} 已存在，内容将写入到 {new_path}')
        path = new_path
    with open(path, 'w', encoding='utf-8') as f:
        for item in data_list:
            f.write(str(item).strip() + '\n')

In [7]:
zh_gt_labs_lines = readListFromFile('./test_wavs/zh_test_lines.jsonl')
en_gt_labs_lines = readListFromFile('./test_wavs/en_test_lines.jsonl')
ns_gt_labs_lines = readListFromFile('./test_wavs/silence_noise_test.jsonl')

audio2lines = {
    line['audio']: line
    for line in zh_gt_labs_lines + en_gt_labs_lines + ns_gt_labs_lines
}

`./zh_our_res.jsonl` 文件中，每一行都是一个 JSON 对象，例如：

```jsonl
{"audio": "00885_00886_00885_701.wav", "total_nonbreak": true, "break_time": -1}
{"audio": "00883_00884_00883_172.wav", "total_nonbreak": false, "break_time": 0.64}
```

### 字段说明

```json
{
  "audio": "音频文件名",
  "total_nonbreak": "是否整段音频都未触发打断信号",
  "break_time": "触发打断信号的时间点（单位：秒）"
}
```

### 规则说明

- `audio`：音频文件名。
- `total_nonbreak`：
  - `true` 表示整段音频没有触发打断信号；
  - `false` 表示音频中触发了打断信号。
- `break_time`：
  - 当 `total_nonbreak = true` 时，`break_time` 通常为 `-1`；
  - 当 `total_nonbreak = false` 时，`break_time` 表示首次触发打断信号的时间点，单位为秒。

---

Each line in `./zh_our_res.jsonl` is a JSON object, for example:

```jsonl
{"audio": "00885_00886_00885_701.wav", "total_nonbreak": true, "break_time": -1}
{"audio": "00883_00884_00883_172.wav", "total_nonbreak": false, "break_time": 0.64}
```

### Field Description

```json
{
  "audio": "Audio filename",
  "total_nonbreak": "Whether the entire audio contains no interruption signal",
  "break_time": "Timestamp of the interruption signal (in seconds)"
}
```

### Rules

- `audio`: the audio filename.
- `total_nonbreak`:
  - `true` means no interruption signal was triggered throughout the entire audio;
  - `false` means an interruption signal was triggered in the audio.
- `break_time`:
  - When `total_nonbreak = true`, `break_time` is usually `-1`;
  - When `total_nonbreak = false`, `break_time` indicates the timestamp of the first interruption signal, in seconds.


In [8]:
test_result_file = './zh_our_res.jsonl'
test_result_lines = readListFromFile(test_result_file)

false_alarm_num = 0
for line in test_result_lines:
    audio = line['audio']
    labs = audio2lines[audio]

    test_break_time = line['break_time']
    test_nonbreak = line['total_nonbreak']

    gt_break_time = labs['break_time']
    gt_nonbreak = labs['total_nonbreak']
    duration = labs['duration']

    line['duration'] = duration
    line['gt_break_time'] = gt_break_time
    line['gt_nonbreak'] = gt_nonbreak

    if test_break_time <= gt_break_time + 0.05 and test_break_time >= gt_break_time - 0.05: #结果打断时间和真实打断时间相近
        line['advance_time'] = 0
        line['delay_time'] = 0
        line['penalty_time'] = 0
        continue
    elif gt_nonbreak and test_nonbreak:   #真实不打断，结果不打断
        line['advance_time'] = 0
        line['delay_time'] = 0
        line['penalty_time'] = 0
        continue
    elif gt_nonbreak and not test_nonbreak: #真实不打断，结果打断
        false_alarm_num += 1
        line['advance_time'] = 0
        line['delay_time'] = 0
        line['penalty_time'] = duration
    elif not gt_nonbreak and test_nonbreak: #真实打断，结果不打断
        line['advance_time'] = 0
        line['delay_time'] = 0
        line['penalty_time'] = (duration-gt_break_time)
    elif not gt_nonbreak and not test_nonbreak: #真实打断，结果打断
        if test_break_time >= gt_break_time:
            line['advance_time'] = 0
            line['delay_time'] = (test_break_time - gt_break_time)
            line['penalty_time'] = (test_break_time - gt_break_time)
        elif test_break_time < gt_break_time:
            false_alarm_num += 1
            line['advance_time'] = 0
            line['delay_time'] = 0
            line['penalty_time'] = duration


In [9]:
delay_items = [line['delay_time'] for line in test_result_lines if line['delay_time'] > 0]
penalty_time_items = [line['penalty_time'] for line in test_result_lines]

print("FIR:",false_alarm_num/len(test_result_lines), "IRL:", sum(delay_items)/len(delay_items), "APT:", sum(penalty_time_items)/len(penalty_time_items))

FIR: 0.1375 IRL: 0.33837436932391457 APT: 0.6561522135416662
